In [17]:
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine
import os
import dotenv
dotenv.load_dotenv()
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

In [18]:
dbms = 'postgresql'
package = 'psycopg'
user = 'postgres'
password = POSTGRES_PASSWORD
host = 'localhost'
port = '5432'
db = 'contrans'

engine = create_engine(f'{dbms}+{package}://{user}:{password}@{host}:{port}/{db}')

In [19]:
myquery = '''
SELECT *
FROM members
WHERE state='Virginia'
'''

pd.read_sql_query(myquery, con=engine)

,bioguide_id,full_name,chamber,state,party,district,birthyear,image,office_address,phone,website,left_right_ideology
0,B001292,"Donald S. Beyer, Jr.",House of Representatives,Virginia,Democratic,8.0,1950.0,https://www.congress.gov/img/member/b001292_20...,"1226 Longworth House Office Building, Washingt...",(202) 225-4376,https://beyer.house.gov,-0.392
1,C001078,Gerald E. Connolly,House of Representatives,Virginia,Democratic,11.0,1950.0,https://www.congress.gov/img/member/c001078_20...,NaN,NaN,NaN,-0.309
2,C001118,Ben Cline,House of Representatives,Virginia,Republican,6.0,1972.0,https://www.congress.gov/img/member/69825a92eb...,"2443 Rayburn House Office Building, Washington...",(202) 225-5431,https://cline.house.gov,0.712
3,G000568,H. Morgan Griffith,House of Representatives,Virginia,Republican,9.0,1958.0,https://www.congress.gov/img/member/68094df86c...,"2110 Rayburn House Office Building, Washington...",(202) 225-3861,https://morgangriffith.house.gov/,0.510
4,K000384,Tim Kaine,Senate,Virginia,Democratic,NaN,1958.0,https://www.congress.gov/img/member/k000384_20...,231 Russell Senate Office Building Washington...,(202) 224-4024,https://www.kaine.senate.gov/,-0.233
5,K000399,Jennifer A. Kiggans,House of Representatives,Virginia,Republican,2.0,1971.0,https://www.congress.gov/img/member/66b0ce45b0...,"152 Cannon House Office Building, Washington, ...",(202) 225-4215,https://kiggans.house.gov,0.264
6,M001227,Jennifer L. McClellan,House of Representatives,Virginia,Democratic,4.0,1972.0,https://www.congress.gov/img/member/m001227_20...,"1628 Longworth House Office Building, Washingt...",(202) 225-6365,https://mcclellan.house.gov/,-0.555
7,M001239,John J. McGuire III,House of Representatives,Virginia,Republican,5.0,1968.0,https://www.congress.gov/img/member/67744ba20b...,"1013 Longworth House Office Building, Washingt...",(202) 225-4711,https://mcguire.house.gov/,0.706
8,S000185,"Robert C. ""Bobby"" Scott",House of Representatives,Virginia,Democratic,3.0,1947.0,https://www.congress.gov/img/member/s000185_20...,"2328 Rayburn House Office Building, Washington...",(202) 225-8351,https://bobbyscott.house.gov,-0.450
9,S001230,Suhas Subramanyam,House of Representatives,Virginia,Democratic,10.0,1986.0,https://www.congress.gov/img/member/6797be8bc7...,"1009 Longworth House Office Building, Washingt...",(202) 225-5136,https://subramanyam.house.gov/,-0.238


Etiquette for SQL

1. Clauses (commands that operate on the data) are CAPS.
2. Data tables and columns are lowercase.
3. Every clause on a new line.
4. If a clause is connected or an extentsion of a previous clause, indent it.

# Look up a bioguideID

In [20]:
name = 'John J. McGuire III'  # from the user's dropdown menu

myquery = '''
SELECT *
FROM members
WHERE full_name = '{name}'
'''

pd.read_sql_query(myquery, con=engine)

,bioguide_id,full_name,chamber,state,party,district,birthyear,image,office_address,phone,website,left_right_ideology


In [21]:
bioguide_id = 'M001239'
myquery = f'''
SELECT
    SUM(c.bioguide_id),
    e.expenditure_amount,
    e.committee_name,
    e.committee_type_full,
    e.support_oppose_indicator
FROM expenditures e
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
WHERE c.bioguide_id = '{bioguide_id}'
GROUP BY e.committee_name
'''

pd.read_sql_query(myquery, con=engine)

DatabaseError: Execution failed on sql '
SELECT
    SUM(c.bioguide_id),
    e.expenditure_amount,
    e.committee_name,
    e.committee_type_full,
    e.support_oppose_indicator
FROM expenditures e
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
WHERE c.bioguide_id = 'M001239'
GROUP BY e.committee_name
': (psycopg.errors.UndefinedFunction) function sum(text) does not exist
LINE 3:     SUM(c.bioguide_id),
            ^
HINT:  No function matches the given name and argument types. You might need to add explicit type casts.
[SQL: 
SELECT
    SUM(c.bioguide_id),
    e.expenditure_amount,
    e.committee_name,
    e.committee_type_full,
    e.support_oppose_indicator
FROM expenditures e
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
WHERE c.bioguide_id = 'M001239'
GROUP BY e.committee_name
]
(Background on this error at: https://sqlalche.me/e/21/f405)

In [ ]:
bioguide_id = 'M001239'
myquery = f'''
SELECT
    SUM(e.expenditure_amount)
        AS expenditure_amount,
    e.committee_name,
    e.support_oppose_indicator
FROM expenditures e
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
WHERE c.bioguide_id = '{bioguide_id}'
GROUP BY e.committee_name, e.support_oppose_indicator
'''

pd.read_sql_query(myquery, con=engine)

,expenditure_amount,committee_name,support_oppose_indicator
0,21924.24,CIVIC RENEWAL FUND INC.,S
1,128860.80,NO GOING BACK PAC INC.,S
2,1160619.62,OUR VIRGINIA,O
3,50591.02,RP PAC,O
4,260771.68,VIRGINIANS FOR CONSERVATIVE LEADERSHIP PAC,S


In [ ]:
myquery = f'''
SELECT
    SUM(e.expenditure_amount)
        AS expenditure_amount,
    e.support_oppose_indicator
FROM expenditures e
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
WHERE c.bioguide_id = '{bioguide_id}'
GROUP BY e.support_oppose_indicator
'''

pd.read_sql_query(myquery, con=engine)

,expenditure_amount,support_oppose_indicator
0,1211210.64,O
1,411556.72,S


# Expenditure committees by ideology of members they donate to

In [ ]:
myquery = '''
SELECT
    m.committee_name,
    SUM(e.expenditure_amount) AS expenditure_amount,
    SUM(m.left_right_ideology * e.expenditure_amount *
    CASE
        WHEN e.support_oppose_indicator = 'S' THEN 1
        WHEN e.support_oppose_indicator = 'O' THEN -1
    END) AS support
FROM expenditures c
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
INNER JOIN members m
    ON c.bioguide_id = m.bioguide_id
GROUP BY m.bioguide_id, e.committee_name
ORDER BY ideology DESC
'''

pd.read_sql_query(myquery, con=engine)

DatabaseError: Execution failed on sql '
SELECT
    m.committee_name,
    SUM(e.expenditure_amount) AS expenditure_amount,
    SUM(m.left_right_ideology * e.expenditure_amount *
    CASE
        WHEN e.support_oppose_indicator = 'S' THEN 1
        WHEN e.support_oppose_indicator = 'O' THEN -1
    END) AS support
FROM expenditures c
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
INNER JOIN members m
    ON c.bioguide_id = m.bioguide_id
GROUP BY m.bioguide_id, e.committee_name
': (psycopg.errors.DuplicateAlias) table name "c" specified more than once
[SQL: 
SELECT
    m.committee_name,
    SUM(e.expenditure_amount) AS expenditure_amount,
    SUM(m.left_right_ideology * e.expenditure_amount *
    CASE
        WHEN e.support_oppose_indicator = 'S' THEN 1
        WHEN e.support_oppose_indicator = 'O' THEN -1
    END) AS support
FROM expenditures c
INNER JOIN crosswalk c
    ON e.fec_id = c.fec_id
INNER JOIN members m
    ON c.bioguide_id = m.bioguide_id
GROUP BY m.bioguide_id, e.committe